# 05 Launch Scaffold Experiments

Goal:

1. define experiment configurations in the notebook
2. generate the shell script automatically
3. `chmod +x`
4. launch in the background with `nohup bash ... > launcher.log 2>&1 &`
5. write a manifest that records the mapping from experiment to output/log/script

This notebook is initially designed for Scaffold verification experiments.


In [ ]:
from pathlib import Path
import json
import os
import stat
import subprocess
from datetime import datetime

CV_ROOT = Path('~/CV_Project').expanduser()
PART1_ROOT = CV_ROOT / 'part1'
SCRIPTS_DIR = PART1_ROOT / 'scripts' / 'train'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part1'
THIRD_PARTY = CV_ROOT / 'third_party'

SCENE = 'Re10k-1'
GROUP = 'scaffold_40k'
GS_MODEL = 'scaffold'   # 'scaffold' or '3dgs'
PLAN = 'verification'

CONDA = Path('~/miniconda3/bin/conda').expanduser()
ENV_NAME = 'scaffold_gs_cu124'
GS_ROOT = THIRD_PARTY / 'Scaffold-GS'
SCRIPT_NAME = 'run_scaffold_40k_verification_generated.sh'
LAUNCHER_LOG = 'scaffold_40k_launcher.log'

DATA_DIR = OUTPUT_ROOT / SCENE / 'verification' / GROUP / 'data'
LOG_DIR = DATA_DIR / 'logs'
MANIFEST_PATH = OUTPUT_ROOT / SCENE / 'verification' / GROUP / 'manifest.json'
SCRIPT_PATH = SCRIPTS_DIR / SCRIPT_NAME
LAUNCHER_LOG_PATH = OUTPUT_ROOT / SCENE / 'verification' / GROUP / LAUNCHER_LOG

for d in [SCRIPTS_DIR, DATA_DIR, LOG_DIR, MANIFEST_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print('SCRIPT_PATH =', SCRIPT_PATH)
print('MANIFEST_PATH =', MANIFEST_PATH)
print('LAUNCHER_LOG_PATH =', LAUNCHER_LOG_PATH)


## 1. Define sources and experiment configurations

This section explicitly records:
- dataset
- plan
- gs model
- source scene path
- argument string
- output path
- log path

This makes launch and result reading share the same mapping.


In [ ]:
SOURCE_MAP = {
    'colmap_full': CV_ROOT / 'dataset' / SCENE / 'part1' / 'planA' / 'colmap_full' / 'gs_scene',
    'colmap_96': CV_ROOT / 'dataset' / SCENE / 'part1' / 'planA' / 'colmap_96' / 'gs_scene',
    'vggt_96': CV_ROOT / 'dataset' / SCENE / 'part1' / 'planB' / 'vggt_96' / 'gs_scene',
}

ITERATIONS = 40000
EVAL_INTERVAL = 2000
ITERS = ' '.join(str(i) for i in range(EVAL_INTERVAL, ITERATIONS + 1, EVAL_INTERVAL))

EXPERIMENTS = [
    {
        'experiment_name': 'colmap_full_vs0_uif8_app0_40k',
        'dataset': SCENE,
        'plan': 'planA',
        'source_key': 'colmap_full',
        'gs_model': 'scaffold',
        'gpu': 0,
        'port': 6010,
        'extra_args': '--voxel_size 0 --update_init_factor 8 --appearance_dim 0',
    },
    {
        'experiment_name': 'colmap_96_vs0_uif8_app0_40k',
        'dataset': SCENE,
        'plan': 'planA',
        'source_key': 'colmap_96',
        'gs_model': 'scaffold',
        'gpu': 0,
        'port': 6010,
        'extra_args': '--voxel_size 0 --update_init_factor 8 --appearance_dim 0',
    },
    {
        'experiment_name': 'vggt_96_vs0_uif8_app0_40k',
        'dataset': SCENE,
        'plan': 'planB',
        'source_key': 'vggt_96',
        'gs_model': 'scaffold',
        'gpu': 1,
        'port': 6011,
        'extra_args': '--voxel_size 0 --update_init_factor 8 --appearance_dim 0',
    },
]

for exp in EXPERIMENTS:
    exp['source_dir'] = str(SOURCE_MAP[exp['source_key']])
    exp['output_dir'] = str(DATA_DIR / exp['experiment_name'])
    exp['log_file'] = str(LOG_DIR / f"{exp['experiment_name']}.log")
    exp['iterations'] = ITERATIONS
    exp['eval_interval'] = EVAL_INTERVAL

EXPERIMENTS


## 2. Generate the shell script


In [ ]:
D = '$'
script_lines = []
script_lines.append('#!/usr/bin/env bash')
script_lines.append('set -euo pipefail')
script_lines.append('')
script_lines.append(f'CONDA="{CONDA}"')
script_lines.append(f'ENV_NAME="{ENV_NAME}"')
script_lines.append(f'GS_ROOT="{GS_ROOT}"')
script_lines.append(f'LOGDIR="{LOG_DIR}"')
script_lines.append(f'ITERS="{ITERS}"')
script_lines.append(f'mkdir -p "{D}LOGDIR"')
script_lines.append('')
script_lines.append('run_train () {')
script_lines.append(f'  local gpu="{D}1"; shift')
script_lines.append(f'  local port="{D}1"; shift')
script_lines.append(f'  local tag="{D}1"; shift')
script_lines.append(f'  local source="{D}1"; shift')
script_lines.append(f'  local model="{D}1"; shift')
script_lines.append(f'  local extra_args="{D}1"; shift')
script_lines.append(f'  local log="{D}LOGDIR/{D}{{tag}}.log"')
script_lines.append(f'  rm -rf "{D}model"')
script_lines.append(f'  echo "[{D}(date '+%F %T')] START {D}tag gpu={D}gpu port={D}port" | tee "{D}log"')
script_lines.append(f'  echo "source={D}source" | tee -a "{D}log"')
script_lines.append(f'  echo "model={D}model" | tee -a "{D}log"')
script_lines.append(f'  echo "extra_args={D}extra_args" | tee -a "{D}log"')
script_lines.append(f'  cd "{D}GS_ROOT"')
script_lines.append(f'  CUDA_VISIBLE_DEVICES="{D}gpu" "{D}CONDA" run -n "{D}ENV_NAME" python train.py \')
script_lines.append(f'    -s "{D}source" \')
script_lines.append(f'    -m "{D}model" \')
script_lines.append('    --eval \')
script_lines.append(f'    --port "{D}port" \')
script_lines.append(f'    --iterations {ITERATIONS} \')
script_lines.append(f'    --test_iterations {D}ITERS \')
script_lines.append(f'    --save_iterations {D}ITERS \')
script_lines.append(f'    {D}extra_args 2>&1 | tee -a "{D}log"')
script_lines.append(f'  echo "[{D}(date '+%F %T')] END {D}tag gpu={D}gpu port={D}port" | tee -a "{D}log"')
script_lines.append('}')
script_lines.append('')

by_gpu = {}
for exp in EXPERIMENTS:
    by_gpu.setdefault(exp['gpu'], []).append(exp)

for gpu, exps in sorted(by_gpu.items()):
    script_lines.append(f'worker_gpu{gpu} () {{')
    for exp in exps:
        script_lines.append(
            f'  run_train {exp["gpu"]} {exp["port"]} {exp["experiment_name"]} "{exp["source_dir"]}" "{exp["output_dir"]}" "{exp["extra_args"]}"'
        )
    script_lines.append('}')
    script_lines.append('')

for gpu in sorted(by_gpu):
    script_lines.append(f'worker_gpu{gpu} &')
    script_lines.append(f'PID{gpu}={D}!')
for gpu in sorted(by_gpu):
    script_lines.append(f'wait "{D}PID{gpu}"')
script_lines.append(f'echo "[{D}(date '+%F %T')] ALL DONE"')

script_text = '
'.join(script_lines) + '
'
SCRIPT_PATH.write_text(script_text, encoding='utf-8')
print(script_text[:4000])


## 3. chmod +x and write the manifest


In [ ]:
SCRIPT_PATH.chmod(SCRIPT_PATH.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

manifest = {
    'scene': SCENE,
    'group': GROUP,
    'plan': PLAN,
    'gs_model': GS_MODEL,
    'script_file': str(SCRIPT_PATH),
    'launcher_log': str(LAUNCHER_LOG_PATH),
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'experiments': EXPERIMENTS,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('wrote manifest to', MANIFEST_PATH)
print('chmod +x done for', SCRIPT_PATH)


## 4. Launch in the background


In [ ]:
launch_cmd = f'nohup bash "{SCRIPT_PATH}" > "{LAUNCHER_LOG_PATH}" 2>&1 &'
print('Launch command:')
print(launch_cmd)

# Uncomment to actually launch:
# subprocess.run(launch_cmd, shell=True, check=True)

print('When launched, the launcher log will be:', LAUNCHER_LOG_PATH)
